# Score-CAM vs GradCAM / LayerCAM — MoNuSAC 2020

This notebook adds **Score-CAM** [Wang et al., 2020] to the MoNuSAC 2020 evaluation,
comparing it against the four gradient-based methods already evaluated in
`pannuke_monusac_cam.ipynb`.

**Prerequisites:** `pannuke_monusac_cam.ipynb` must have been run first to produce:
- `outputs/monusac_cam_comparison/best_densenet169_monusac.pth`
- `outputs/monusac_cam_comparison/results_50_test.csv`  (or `results_test.csv`)

**Design note — no FPN-ScoreCAM:**  
FPN-ScoreCAM is deferred to future work for the same reasons as on PanNuke:
early-layer channel contributions require full downstream propagation and cannot
be blended post-hoc. The scientific question (pyramid resolution-dependence) is
already answered by the gradient-based FPN results on MoNuSAC.

**MoNuSAC-specific differences from the PanNuke ScoreCAM notebook:**
- Images and masks are loaded from in-memory numpy arrays (`all_images`, `all_masks`)
  rather than from saved files, so the data-loading cells of
  `pannuke_monusac_cam.ipynb` must be re-run here.
- Test indices come from `test_cam_indices` (array of global indices), not a JSON.
- Masks are 4-channel `(H, W, 4)` float32, not 6-channel.
- `CLASS_NAMES` = `['Epithelial','Lymphocyte','Macrophage','Neutrophil']`.
- Class colours are `CLASS_COLORS` (list of hex), not `CLASS_COLORS_F32`.

## 1. Imports & Environment

In [ ]:
import gc
import json
import random
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration
All constants are identical to `pannuke_monusac_cam.ipynb`.
`scorecam` is added to the method registry with purple colour.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
MONUSAC_ROOT = Path('MoNuSAC_images_and_annotations')  # adjust if needed
OUTPUT_ROOT  = Path('./outputs')
GRAD_OUT_DIR = OUTPUT_ROOT / 'monusac_cam_comparison'
MODEL_PATH   = GRAD_OUT_DIR / 'best_densenet169_monusac.pth'

OUT_DIR  = OUTPUT_ROOT / 'monusac_scorecam'
FIG_DIR  = OUT_DIR / 'figures'
TRIP_DIR = OUT_DIR / 'triplets'
for d in [OUT_DIR, FIG_DIR, TRIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Class definitions (identical to main notebook) ────────────────────────────
CLASS_NAMES:   List[str] = ['Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil']
CLASS_DISPLAY: List[str] = ['Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil']
NUM_CLASSES = 4

XML_CLASS_MAP: Dict[str, int] = {
    'Epithelial': 0, 'epithelial': 0, 'EPITHELIAL': 0,
    'Lymphocyte': 1, 'lymphocyte': 1, 'LYMPHOCYTE': 1,
    'Macrophage': 2, 'macrophage': 2, 'MACROPHAGE': 2,
    'Neutrophil': 3, 'neutrophil': 3, 'NEUTROPHIL': 3,
    'Ambiguous': -1, 'ambiguous': -1,
}

OFFICIAL_TEST_PATIENTS: List[str] = [
    'TCGA-55-1594-01Z-00-DX1', 'TCGA-67-3771-01A-01-BS1',
    'TCGA-69-7760-01A-01-BS1', 'TCGA-73-4668-01A-01-BS1',
    'TCGA-EJ-5510-01A-01-BS1', 'TCGA-G9-6336-01A-01-BS1',
    'TCGA-G9-6348-01A-01-BS1', 'TCGA-VP-A878-01A-01-BS1',
    'TCGA-A2-A0CV-01A-02-TSB', 'TCGA-A7-A0CE-01A-11-TS2',
    'TCGA-D8-A1JF-01A-01-TS1', 'TCGA-E2-A14N-01A-02-TSB',
    'TCGA-2Z-A9J9-01A-01-TS1', 'TCGA-B0-5698-01A-01-BS1',
    'TCGA-B0-5710-01A-01-BS1', 'TCGA-B0-5711-01A-01-BS1',
]

# ── Shared evaluation constants ───────────────────────────────────────────────
IMG_SIZE     = 256
THRESHOLD    = 0.5
IOU_THRS     = [0.25, 0.40, 0.50]
NOISE_THR    = 0.25
ALPHA_SCALE  = 0.85
BATCH_SIZE   = 16
NUM_EPOCHS   = 30
LR_HEAD      = 1e-4
LR_BACKBONE  = 1e-5
WEIGHT_DECAY = 1e-4
DROPOUT_P    = 0.3
GRAD_CLIP    = 1.0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Gradient method registry (existing 4 methods) ────────────────────────────
GRAD_METHODS = ['std_gradcam', 'fpn_gradcam', 'std_layercam', 'fpn_layercam']

# ── All 5 methods including Score-CAM ─────────────────────────────────────────
ALL_METHODS = GRAD_METHODS + ['scorecam']

METHOD_LABELS = {
    'std_gradcam'  : 'Standard GradCAM',
    'fpn_gradcam'  : 'FPN-GradCAM',
    'std_layercam' : 'Standard LayerCAM',
    'fpn_layercam' : 'FPN-LayerCAM',
    'scorecam'     : 'Score-CAM',
}
METHOD_COLORS = {
    'std_gradcam'  : '#90CAF9',
    'fpn_gradcam'  : '#EF9A9A',
    'std_layercam' : '#A5D6A7',
    'fpn_layercam' : '#FFE082',
    'scorecam'     : '#CE93D8',   # purple — gradient-free
}
METHOD_MARKERS = {
    'std_gradcam'  : 'o',
    'fpn_gradcam'  : 's',
    'std_layercam' : '^',
    'fpn_layercam' : 'D',
    'scorecam'     : 'P',
}

# Class colours for overlay (hex list — same as main notebook)
CLASS_COLORS = ['#E53935', '#1E88E5', '#FFB300', '#43A047']
CLASS_COLORS_F32 = [
    (int(h[1:3],16)/255, int(h[3:5],16)/255, int(h[5:7],16)/255)
    for h in CLASS_COLORS
]

# ── FPN config (kept for reference; not used by Score-CAM) ────────────────────
FPN_LAYERS: List[Tuple[str, float]] = [
    ('backbone.features.denseblock1', 0.20),
    ('backbone.features.denseblock2', 0.35),
    ('backbone.features.denseblock4', 0.45),
]

# ── Score-CAM settings ────────────────────────────────────────────────────────
SCORECAM_LAYER = 'backbone.features.denseblock4'  # 1,664 channels
SCORECAM_BATCH = 64   # forward passes per GPU batch; reduce to 32 if OOM

print('Configuration loaded.')
print(f'Score-CAM layer : {SCORECAM_LAYER}  ({SCORECAM_BATCH} channels/batch)')
print(f'Output dir      : {OUT_DIR}')

## 3. XML Parser and Data Loading

Exact copy of the data-loading pipeline from `pannuke_monusac_cam.ipynb`.
This re-loads images and masks into memory so this notebook is fully self-contained.

In [ ]:
def parse_monusac_xml(xml_path: Path, img_h: int, img_w: int) -> np.ndarray:
    """Parse MoNuSAC XML (Attribute-based format) → (H, W, 4) uint8 mask."""
    mask = np.zeros((img_h, img_w, NUM_CLASSES), dtype=np.uint8)
    try:
        root = ET.parse(str(xml_path)).getroot()
    except ET.ParseError:
        return mask

    for ann in root.findall('Annotation'):
        class_idx = -1
        attrs_block = ann.find('Attributes')
        if attrs_block is not None:
            first_attr = attrs_block.find('Attribute')
            if first_attr is not None:
                raw  = first_attr.get('Name', '').strip()
                class_idx = XML_CLASS_MAP.get(raw,
                            XML_CLASS_MAP.get(raw.capitalize(), -1))
        if class_idx < 0:
            continue

        for region in ann.findall('.//Region'):
            vertices = region.findall('.//Vertex')
            if len(vertices) < 3:
                continue
            pts = []
            for v in vertices:
                x = v.get('X') or v.get('x')
                y = v.get('Y') or v.get('y')
                if x is not None and y is not None:
                    pts.append([float(x), float(y)])
            if len(pts) < 3:
                continue
            pts_arr = np.array(pts, dtype=np.float32)
            pts_arr[:, 0] = np.clip(pts_arr[:, 0], 0, img_w - 1)
            pts_arr[:, 1] = np.clip(pts_arr[:, 1], 0, img_h - 1)
            pts_int = pts_arr.astype(np.int32).reshape((-1, 1, 2))
            channel = np.ascontiguousarray(mask[..., class_idx])
            cv2.fillPoly(channel, [pts_int], color=1)
            mask[..., class_idx] = channel
    return mask


def load_monusac_sample(
    tif_path: Path, xml_path: Path, target_size: int = IMG_SIZE,
) -> Tuple[np.ndarray, np.ndarray]:
    img_pil  = Image.open(tif_path).convert('RGB')
    orig_w, orig_h = img_pil.size
    mask_orig = parse_monusac_xml(xml_path, orig_h, orig_w).astype(np.float32)
    img_resized = np.array(
        img_pil.resize((target_size, target_size), Image.BILINEAR), dtype=np.uint8
    )
    if orig_h != target_size or orig_w != target_size:
        mask_resized = np.stack([
            cv2.resize(mask_orig[..., c], (target_size, target_size),
                       interpolation=cv2.INTER_NEAREST)
            for c in range(NUM_CLASSES)
        ], axis=-1)
    else:
        mask_resized = mask_orig
    return img_resized, mask_resized.astype(np.float32)


def build_monusac_index(root: Path, verbose: bool = True) -> pd.DataFrame:
    records = []
    for pdir in sorted(d for d in root.iterdir() if d.is_dir()):
        pid     = pdir.name
        is_test = pid in OFFICIAL_TEST_PATIENTS
        for tif in sorted(pdir.glob('*.tif')):
            xml = tif.with_suffix('.xml')
            if xml.exists():
                records.append({'patient_id': pid, 'sample_name': tif.stem,
                                'tif_path': str(tif), 'xml_path': str(xml),
                                'is_test': is_test})
    df = pd.DataFrame(records)
    if verbose:
        print(f'Total: {len(df)}  test: {df.is_test.sum()}  train: {(~df.is_test).sum()}')
    return df


def load_all_samples(df: pd.DataFrame, target_size: int = IMG_SIZE):
    N = len(df)
    images = np.zeros((N, target_size, target_size, 3), dtype=np.uint8)
    masks  = np.zeros((N, target_size, target_size, NUM_CLASSES), dtype=np.float32)
    for i, row in enumerate(tqdm(df.itertuples(), total=N, desc='Loading')):
        img, msk = load_monusac_sample(
            Path(row.tif_path), Path(row.xml_path), target_size)
        images[i] = img
        masks[i]  = msk
    return images, masks


print('Scanning MoNuSAC directory...')
df_index = build_monusac_index(MONUSAC_ROOT)

# Fallback to random 80/20 if no official test patients matched
if df_index['is_test'].sum() == 0:
    rng_split    = np.random.default_rng(SEED)
    all_patients = df_index['patient_id'].unique().copy()
    rng_split.shuffle(all_patients)
    n_test_pts   = max(1, int(len(all_patients) * 0.2))
    test_pts     = set(all_patients[:n_test_pts])
    df_index['is_test'] = df_index['patient_id'].isin(test_pts)
    print(f'Fallback split: {df_index.is_test.sum()} test / {(~df_index.is_test).sum()} train')

In [ ]:
print('Loading all MoNuSAC images and masks into memory...')
all_images, all_masks = load_all_samples(df_index)
all_labels = (all_masks.sum(axis=(1, 2)) > 0).astype(np.uint8)  # (N, 4)

print(f'Images : {all_images.shape}  {all_images.dtype}')
print(f'Masks  : {all_masks.shape}   {all_masks.dtype}')
print(f'Labels : {all_labels.shape}  {all_labels.dtype}')
print('\nCardinality distribution:')
print(pd.Series(all_labels.sum(axis=1)).value_counts().sort_index())

In [ ]:
# ── Test / train / val split (identical to main notebook) ─────────────────────
test_mask_bool  = df_index['is_test'].values
train_mask_bool = ~test_mask_bool
test_global_indices = np.where(test_mask_bool)[0]

rng_tv = np.random.default_rng(SEED)
train_patients = df_index.loc[train_mask_bool, 'patient_id'].unique()
rng_tv.shuffle(train_patients)
n_val_pts = max(1, int(len(train_patients) * 0.2))
val_pts   = set(train_patients[:n_val_pts])
train_pts = set(train_patients[n_val_pts:])

train_indices = np.where(df_index['patient_id'].isin(train_pts) & train_mask_bool)[0]
val_indices   = np.where(df_index['patient_id'].isin(val_pts)   & train_mask_bool)[0]

# Use ALL test images (Comment 1 fix applied)
test_cam_indices = np.sort(test_global_indices.copy())
N_TEST = len(test_cam_indices)

print(f'Train : {len(train_indices):,}')
print(f'Val   : {len(val_indices):,}')
print(f'Test  : {N_TEST}  (all test-split images)')

## 4. Model

In [ ]:
class DenseNet169MultiLabel(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_p: float = 0.3) -> None:
        super().__init__()
        backbone = models.densenet169(weights=None)
        in_features = backbone.classifier.in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x))


model = DenseNet169MultiLabel(num_classes=NUM_CLASSES, dropout_p=DROPOUT_P).to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint — epoch {ckpt["epoch"]}, val AUC {ckpt["val_auc"]:.4f}')

## 5. Preprocessing and Inference Utilities

In [ ]:
_infer_tfm = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def preprocess(img_rgb: np.ndarray) -> torch.Tensor:
    """HWC uint8 → (1,3,H,W) float32 on DEVICE (no grad — ScoreCAM is gradient-free)."""
    return _infer_tfm(img_rgb).unsqueeze(0).to(DEVICE)


@torch.no_grad()
def get_pred(img_rgb: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (binary_labels, sigmoid_probs) for one image."""
    logits = model(preprocess(img_rgb))
    probs  = torch.sigmoid(logits).squeeze().cpu().numpy()
    return (probs >= THRESHOLD).astype(np.uint8), probs


def compute_iou(cam: np.ndarray, gt_ch: np.ndarray, thr: float) -> float:
    gt_bin = gt_ch > 0
    if not gt_bin.any():
        return float('nan')
    pred  = cam >= thr
    inter = (pred & gt_bin).sum()
    union = (pred | gt_bin).sum()
    return float(inter) / float(union) if union > 0 else float('nan')


def resize_ch(
    ch: np.ndarray, hw: Tuple[int, int] = (IMG_SIZE, IMG_SIZE),
) -> np.ndarray:
    if ch.shape == hw:
        return ch
    return cv2.resize(ch, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)


def render_gt_mask(
    mask_4ch: np.ndarray, hw: Tuple[int, int] = (IMG_SIZE, IMG_SIZE),
) -> np.ndarray:
    H, W   = hw
    canvas = np.zeros((H, W, 3), dtype=np.float32)
    oh, ow = mask_4ch.shape[:2]
    if (oh, ow) != (H, W):
        mask_4ch = np.stack([
            cv2.resize(mask_4ch[..., c], (W, H), interpolation=cv2.INTER_NEAREST)
            for c in range(NUM_CLASSES)
        ], axis=-1)
    for c_idx, color in enumerate(CLASS_COLORS_F32):
        canvas[mask_4ch[..., c_idx] > 0] = color
    return (canvas * 255).clip(0, 255).astype(np.uint8)


def build_cam_overlay(
    cam: np.ndarray, class_idx: int, orig_resized: np.ndarray,
) -> np.ndarray:
    """Single-class CAM overlay."""
    r, g, b = CLASS_COLORS_F32[class_idx]
    clean   = np.where(cam >= NOISE_THR, cam, 0.0)
    comp_c  = np.stack([clean*r, clean*g, clean*b], axis=-1)
    alpha   = (clean * ALPHA_SCALE)[..., None]
    bg      = orig_resized.astype(np.float32) / 255.0
    return ((comp_c * alpha + bg * (1 - alpha)).clip(0, 1) * 255).astype(np.uint8)


print('Utilities defined.')

## 6. Score-CAM Engine

Identical implementation to the PanNuke Score-CAM notebook.
No gradient computation — importance weights come from causal masking.

In [ ]:
class ScoreCAMEngine:
    """Gradient-free Score-CAM for DenseNet169 multi-label classification.

    Parameters
    ----------
    model       : trained DenseNet169MultiLabel in eval mode
    layer_path  : dotted path to target conv layer
    batch_size  : channels processed per GPU batch
    """

    def __init__(
        self,
        model      : nn.Module,
        layer_path : str = SCORECAM_LAYER,
        batch_size : int = SCORECAM_BATCH,
    ) -> None:
        self.model      = model
        self.batch_size = batch_size
        self._acts: Optional[torch.Tensor] = None

        layer = model
        for k in layer_path.split('.'):
            layer = getattr(layer, k)

        self._handle = layer.register_forward_hook(
            lambda m, i, o: setattr(self, '_acts', o.detach())
        )

    def remove_hooks(self) -> None:
        self._handle.remove()

    @torch.no_grad()
    def generate(
        self,
        img_tensor: torch.Tensor,  # (1, 3, H, W) on DEVICE — no requires_grad needed
        class_idx : int,
    ) -> np.ndarray:               # (H, W) float32 in [0, 1]
        """Generate Score-CAM heatmap for one class."""
        H, W = img_tensor.shape[2], img_tensor.shape[3]

        # Step 1: baseline forward pass → activation maps
        _ = self.model(img_tensor)
        acts = self._acts.squeeze(0)    # (K, h, w)
        K, h, w = acts.shape

        # Step 2: upsample + per-channel min-max normalise
        acts_up = F.interpolate(
            acts.unsqueeze(0), size=(H, W),
            mode='bilinear', align_corners=False,
        ).squeeze(0)                    # (K, H, W)

        a_min = acts_up.flatten(1).min(dim=1).values.view(K, 1, 1)
        a_max = acts_up.flatten(1).max(dim=1).values.view(K, 1, 1)
        masks = (acts_up - a_min) / (a_max - a_min).clamp(min=1e-8)  # (K,H,W)

        # Step 3: masked forward passes in batches → confidence scores
        scores = torch.zeros(K, device=DEVICE)
        for start in range(0, K, self.batch_size):
            end        = min(start + self.batch_size, K)
            batch_k    = end - start
            img_batch  = img_tensor.expand(batch_k, -1, -1, -1)
            mask_batch = masks[start:end].unsqueeze(1)   # (batch_k,1,H,W)
            logits     = self.model(img_batch * mask_batch)
            scores[start:end] = torch.sigmoid(logits[:, class_idx])

        # Step 4: weighted sum of activation maps → heatmap
        w_k     = scores.view(K, 1, 1)
        cam_raw = F.relu((w_k * acts).sum(dim=0, keepdim=True))
        cam_up  = F.interpolate(
            cam_raw.unsqueeze(0), size=(H, W),
            mode='bilinear', align_corners=False,
        ).squeeze().cpu().numpy()

        cam_up = np.maximum(cam_up, 0)
        if cam_up.max() > 1e-8:
            cam_up /= cam_up.max()
        return cam_up.astype(np.float32)


scorecam_engine = ScoreCAMEngine(model, SCORECAM_LAYER, SCORECAM_BATCH)
print('ScoreCAMEngine ready.')
print(f'  Layer  : {SCORECAM_LAYER}  (1,664 channels)')
print(f'  Batch  : {SCORECAM_BATCH} channels/batch')
print(f'  Passes : ~{N_TEST * 2 * 1664:,}  ({N_TEST} imgs × ~2 active classes × 1,664)')

## 7. Run Score-CAM on All Test Images

In [ ]:
scorecam_rows: List[Dict] = []

print(f'Running Score-CAM on {N_TEST} MoNuSAC test images...')

for local_i, gidx in enumerate(tqdm(test_cam_indices, desc='Score-CAM')):
    orig_rgb  = all_images[gidx]       # (H, W, 3) uint8
    mask_4ch  = all_masks[gidx]        # (H, W, 4) float32
    gt_labels = all_labels[gidx]       # (4,) uint8

    pred_lbl, probs = get_pred(orig_rgb)
    active = [i for i, p in enumerate(pred_lbl) if p == 1]

    orig_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))
    gt_mask_rgb  = render_gt_mask(mask_4ch)
    img_t        = preprocess(orig_rgb)   # (1, 3, H, W)

    row: Dict = {
        'image_idx'  : local_i,
        'global_idx' : int(gidx),
        'patient_id' : df_index.iloc[gidx]['patient_id'],
        'sample_name': df_index.iloc[gidx]['sample_name'],
        'cardinality': int(gt_labels.sum()),
        'is_test'    : bool(df_index.iloc[gidx]['is_test']),
    }
    for ci, cname in enumerate(CLASS_NAMES):
        row[f'gt_{cname}']   = int(gt_labels[ci])
        row[f'pred_{cname}'] = int(pred_lbl[ci])
        row[f'prob_{cname}'] = round(float(probs[ci]), 4)

    # ── Generate Score-CAM for each predicted-positive class ─────────────────
    cams_sc: Dict[int, np.ndarray] = {}
    for c_idx in active:
        cam    = scorecam_engine.generate(img_t, c_idx)
        cams_sc[c_idx] = cam
        cname  = CLASS_NAMES[c_idx]
        gt_ch  = resize_ch(mask_4ch[..., c_idx])
        for thr in IOU_THRS:
            row[f'scorecam_iou_{thr:.2f}_{cname}'] = round(
                compute_iou(cam, gt_ch, thr), 4)
        row[f'scorecam_area_{cname}'] = round(
            float((cam >= NOISE_THR).mean()), 4)

    # Fill NaN for non-active classes
    for cname in CLASS_NAMES:
        for thr in IOU_THRS:
            if f'scorecam_iou_{thr:.2f}_{cname}' not in row:
                row[f'scorecam_iou_{thr:.2f}_{cname}'] = float('nan')
        if f'scorecam_area_{cname}' not in row:
            row[f'scorecam_area_{cname}'] = float('nan')

    scorecam_rows.append(row)

    # ── Triplet figure (first active class) ───────────────────────────────────
    if active:
        c_show  = active[0]
        overlay = build_cam_overlay(cams_sc[c_show], c_show, orig_resized)
        iou_v   = row.get(f'scorecam_iou_0.50_{CLASS_NAMES[c_show]}', float('nan'))
        fig = plt.figure(figsize=(16, 5.6), facecolor='#0d0d0d')
        gs  = fig.add_gridspec(1, 3, wspace=0.04)
        for col_i, (panel, title) in enumerate([
            (orig_resized, 'Original Image'),
            (gt_mask_rgb,  'Ground-Truth Mask'),
            (overlay,      'Score-CAM Overlay'),
        ]):
            ax = fig.add_subplot(gs[0, col_i])
            ax.imshow(panel); ax.axis('off')
            ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=4)
        r2, g2, b2 = CLASS_COLORS_F32[c_show]
        marker = '\u2713' if bool(gt_labels[c_show]) else '\u2717'
        iou_s  = f'{iou_v:.3f}' if iou_v == iou_v else 'N/A'
        fig.axes[2].legend(
            handles=[mpatches.Patch(
                facecolor=(r2, g2, b2), edgecolor='white', linewidth=0.4,
                label=f'{marker} {CLASS_DISPLAY[c_show]}  '
                      f'p={probs[c_show]:.2f}  IoU={iou_s}',
            )],
            loc='lower right', fontsize=8, framealpha=0.82,
            facecolor='#111111', edgecolor='#555555', labelcolor='white',
        )
        fig.suptitle(
            f'MoNuSAC #{local_i:03d}  |  Score-CAM  |  '
            f'\u2713=TP  \u2717=FP  patient={df_index.iloc[gidx]["patient_id"][:18]}',
            color='#dddddd', fontsize=9, fontweight='bold', y=1.02,
        )
        plt.savefig(TRIP_DIR / f'scorecam_{local_i:03d}.png',
                    dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
        plt.close(fig)

    if DEVICE.type == 'cuda' and local_i % 20 == 0:
        torch.cuda.empty_cache()

scorecam_engine.remove_hooks()

df_sc = pd.DataFrame(scorecam_rows)
df_sc.to_csv(OUT_DIR / 'scorecam_results_test.csv', index=False)
print(f'Score-CAM results saved. Shape: {df_sc.shape}')

## 8. Merge with Gradient-Method Results

In [ ]:
grad_csv_candidates = [
    GRAD_OUT_DIR / 'results_50_test.csv',
    GRAD_OUT_DIR / 'results_test.csv',
]
grad_csv = next((p for p in grad_csv_candidates if p.exists()), None)
if grad_csv is None:
    raise FileNotFoundError(
        'Gradient-method results CSV not found. '
        'Run pannuke_monusac_cam.ipynb first.'
    )

df_grad = pd.read_csv(grad_csv)
print(f'Gradient results: {df_grad.shape}  from {grad_csv.name}')

# Merge on global_idx — common key between both DataFrames
sc_cols = ['global_idx'] + [
    c for c in df_sc.columns if c.startswith('scorecam')
]
df_all = df_grad.merge(df_sc[sc_cols], on='global_idx', how='inner')
df_all.to_csv(OUT_DIR / 'scorecam_results_all_test.csv', index=False)
print(f'Merged 5-method DataFrame: {df_all.shape}')

## 9. IoU Summary — All 5 Methods

In [ ]:
summary_rows = []
for mkey in ALL_METHODS:
    for thr in IOU_THRS:
        thr_s = f'{thr:.2f}'
        class_means = []
        for cname in CLASS_NAMES:
            col = f'{mkey}_iou_{thr_s}_{cname}'
            if col in df_all.columns:
                class_means.append(df_all[col].dropna().mean())
        summary_rows.append({
            'method'   : METHOD_LABELS[mkey],
            'threshold': thr,
            'macro_iou': round(float(np.nanmean(class_means)), 4),
            **{f'iou_{c}': round(float(v), 4)
               for c, v in zip(CLASS_DISPLAY, class_means)},
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'scorecam_iou_summary.csv', index=False)

df50 = df_summary[df_summary['threshold'] == 0.50]
print(f'IoU@0.50 — All 5 Methods (n={len(df_all)} test images):')
print(f'{"Method":<28}  {"Macro":>7}', end='')
for cd in CLASS_DISPLAY:
    print(f'  {cd[:12]:>12}', end='')
print()
print('-' * 85)
for _, r in df50.iterrows():
    print(f'{r["method"]:<28}  {r["macro_iou"]:>7.4f}', end='')
    for cname in CLASS_NAMES:
        val = r.get(f'iou_{cname}', float('nan'))
        print(f'  {val:>12.4f}', end='')
    print()

sc_macro  = df50[df50['method'] == 'Score-CAM']['macro_iou'].values[0]
best_grad = df50[df50['method'] != 'Score-CAM']['macro_iou'].max()
best_name = df50.loc[df50['macro_iou'] == best_grad, 'method'].values[0]
print(f'\nScore-CAM vs best gradient method ({best_name}): {sc_macro - best_grad:+.4f}')

## 10. Figure 1 — IoU Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, len(IOU_THRS), figsize=(22, 6), sharey=False)
fig.suptitle(
    f'Score-CAM vs GradCAM/LayerCAM variants — MoNuSAC 2020 ({N_TEST} test images)\n'
    'Purple = Score-CAM (gradient-free); other colours = gradient-based',
    fontsize=12, fontweight='bold',
)
x       = np.arange(NUM_CLASSES)
n_m     = len(ALL_METHODS)
w       = 0.15
offsets = np.linspace(-(n_m-1)/2*w, (n_m-1)/2*w, n_m)

for ax, thr in zip(axes, IOU_THRS):
    for mkey, off in zip(ALL_METHODS, offsets):
        means = []
        for cname in CLASS_NAMES:
            col = f'{mkey}_iou_{thr:.2f}_{cname}'
            means.append(df_all[col].dropna().mean()
                         if col in df_all.columns else np.nan)
        edge = '#555555' if mkey == 'scorecam' else 'white'
        lw   = 1.2      if mkey == 'scorecam' else 0.4
        bars = ax.bar(x + off, means, w,
                      label=METHOD_LABELS[mkey],
                      color=METHOD_COLORS[mkey],
                      edgecolor=edge, linewidth=lw, alpha=0.92)
        for b, v in zip(bars, means):
            if not np.isnan(v):
                ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.002,
                        f'{v:.3f}', ha='center', va='bottom',
                        fontsize=4.8, rotation=90)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_DISPLAY, rotation=20, ha='right', fontsize=9)
    ax.set_title(f'IoU @ {thr:.2f}', fontsize=10)
    ax.set_ylabel('Mean IoU')
    ax.set_ylim(0, 0.30)
    ax.legend(fontsize=6, loc='upper right', framealpha=0.6)
    ax.grid(axis='y', alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig1_iou_5methods.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 1 saved.')

## 11. Figure 2 — Macro IoU Heatmap

In [ ]:
hm_data = df50.set_index('method')[
    [f'iou_{c}' for c in CLASS_DISPLAY]
].astype(float)
hm_data.columns = CLASS_DISPLAY

fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                          gridspec_kw={'width_ratios': [2.2, 1]})
fig.suptitle(f'IoU@0.50 — 5 Methods × 4 Classes  (n={N_TEST})',
             fontsize=12, fontweight='bold')
sns.heatmap(
    hm_data, ax=axes[0],
    cmap='YlOrRd', annot=True, fmt='.4f', annot_kws={'size': 10},
    linewidths=0.5, linecolor='#cccccc',
    cbar_kws={'label': 'Mean IoU@0.50', 'shrink': 0.85},
)
axes[0].tick_params(axis='x', rotation=20, labelsize=9)
axes[0].tick_params(axis='y', rotation=0,  labelsize=9)

macros = df50.set_index('method')['macro_iou']
clrs   = [METHOD_COLORS.get(
    next((k for k, v in METHOD_LABELS.items() if v == m), 'x'), '#AAAAAA')
    for m in macros.index]
axes[1].barh(macros.index, macros.values, color=clrs,
             edgecolor='white', linewidth=0.4, alpha=0.92)
for i, v in enumerate(macros.values):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
axes[1].set_xlabel('Macro IoU@0.50', fontsize=9)
axes[1].set_xlim(0, max(macros.values) * 1.28)
axes[1].grid(axis='x', alpha=0.25, linestyle=':')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig2_heatmap_5methods.{ext}', bbox_inches='tight', dpi=300)
plt.show()
print('Figure 2 saved.')

## 12. Figure 3 — Per-(image, class) Scatter: Score-CAM vs Each Gradient Method

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
fig.suptitle(
    'Per-(image, class) IoU@0.50: Score-CAM vs each gradient method — MoNuSAC\n'
    'Points above diagonal = Score-CAM higher; below = gradient method higher',
    fontsize=11, fontweight='bold',
)
for ax, gkey in zip(axes, GRAD_METHODS):
    sc_vals, gr_vals = [], []
    for cname in CLASS_NAMES:
        sc_col = f'scorecam_iou_0.50_{cname}'
        gr_col = f'{gkey}_iou_0.50_{cname}'
        if sc_col in df_all.columns and gr_col in df_all.columns:
            mask = df_all[sc_col].notna() & df_all[gr_col].notna()
            sc_vals.extend(df_all.loc[mask, sc_col].values)
            gr_vals.extend(df_all.loc[mask, gr_col].values)

    sc_arr = np.array(sc_vals)
    gr_arr = np.array(gr_vals)
    if len(sc_arr) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes)
        continue
    ax.scatter(gr_arr, sc_arr, alpha=0.35, s=10,
               color=METHOD_COLORS[gkey], edgecolors='none')
    lim = max(sc_arr.max(), gr_arr.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8, alpha=0.5)
    win_sc = int((sc_arr > gr_arr).sum())
    win_gr = int((gr_arr > sc_arr).sum())
    ax.set_xlabel(f'{METHOD_LABELS[gkey]} IoU@0.50', fontsize=9)
    ax.set_ylabel('Score-CAM IoU@0.50', fontsize=9)
    ax.set_title(
        f'{METHOD_LABELS[gkey]}\n'
        f'Score-CAM wins {win_sc}/{len(sc_arr)}  |  {METHOD_LABELS[gkey]} wins {win_gr}/{len(sc_arr)}',
        fontsize=8.5,
    )
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.grid(alpha=0.2, linestyle=':'); ax.set_aspect('equal')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig3_scatter_scorecam_vs_gradient.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()
print('Figure 3 saved.')

## 13. IoU vs Label Cardinality — Score-CAM vs Gradient Methods

In [ ]:
def pooled_iou_by_card(df: pd.DataFrame, mkey: str, thr: float = 0.50) -> pd.DataFrame:
    thr_s = f'{thr:.2f}'
    rows  = []
    for card in sorted(df['cardinality'].unique()):
        sub  = df[df['cardinality'] == card]
        cols = [f'{mkey}_iou_{thr_s}_{c}' for c in CLASS_NAMES
                if f'{mkey}_iou_{thr_s}_{c}' in df.columns]
        vals = np.concatenate([sub[c].dropna().values for c in cols]) if cols else np.array([])
        if len(vals) == 0:
            continue
        rows.append({
            'cardinality': card,
            'mean'       : float(np.nanmean(vals)),
            'sem'        : float(np.nanstd(vals) / max(np.sqrt(len(vals)), 1)),
        })
    return pd.DataFrame(rows)


fig, ax = plt.subplots(figsize=(11, 6))
fig.suptitle(
    f'IoU@0.50 vs Label Cardinality — All 5 Methods — MoNuSAC (n={len(df_all)})\n'
    'Score-CAM (purple) vs four gradient-based methods',
    fontsize=12, fontweight='bold',
)

spearman_results = {}
for mkey in ALL_METHODS:
    cd = pooled_iou_by_card(df_all, mkey)
    if len(cd) < 2:
        continue
    rho, p = stats.spearmanr(cd['cardinality'], cd['mean'])
    spearman_results[mkey] = (rho, p)
    lw = 2.5 if mkey == 'scorecam' else 1.8
    ax.errorbar(
        cd['cardinality'], cd['mean'],
        yerr=cd['sem'] * 1.96,
        fmt=f"{METHOD_MARKERS[mkey]}-",
        color=METHOD_COLORS[mkey],
        linewidth=lw, markersize=9, capsize=5,
        label=f"{METHOD_LABELS[mkey]} (\u03c1={rho:.3f})",
        zorder=3 if mkey == 'scorecam' else 2,
    )

ax.set_xlabel('Label Cardinality', fontsize=11)
ax.set_ylabel('Mean IoU@0.50 (± 95% CI)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.25, linestyle=':')
ax.set_xticks(sorted(df_all['cardinality'].unique()))

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig4_cardinality_5methods.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()

print('Spearman cardinality test:')
for mkey, (rho, p) in spearman_results.items():
    print(f'  {METHOD_LABELS[mkey]:<28}: rho={rho:+.4f}  p={p:.4f}')

## 14. Pairwise Interference — Score-CAM

In [ ]:
def build_interference_sc(
    df: pd.DataFrame, mkey: str, thr: float = 0.50, min_n: int = 3,
) -> pd.DataFrame:
    thr_s  = f'{thr:.2f}'
    detail = []
    for c in range(NUM_CLASSES):
        cname   = CLASS_NAMES[c]
        iou_col = f'{mkey}_iou_{thr_s}_{cname}'
        if iou_col not in df.columns:
            continue
        base = df[
            (df[f'gt_{cname}'] == 1) & (df[f'pred_{cname}'] == 1)
        ][[iou_col] + [f'gt_{CLASS_NAMES[cp]}' for cp in range(NUM_CLASSES)]].dropna()

        for cp in range(NUM_CLASSES):
            if c == cp:
                continue
            cname_p = CLASS_NAMES[cp]
            absent  = base[base[f'gt_{cname_p}'] == 0][iou_col].values
            present = base[base[f'gt_{cname_p}'] == 1][iou_col].values
            if len(absent) < min_n or len(present) < min_n:
                continue
            delta   = float(np.nanmean(absent)) - float(np.nanmean(present))
            _, pval = stats.ttest_ind(absent, present, equal_var=False,
                                       nan_policy='omit')
            detail.append({
                'method'     : METHOD_LABELS[mkey],
                'class_c'    : CLASS_DISPLAY[c],
                'class_cp'   : CLASS_DISPLAY[cp],
                'delta_iou'  : round(delta, 4),
                'n_absent'   : len(absent),
                'n_present'  : len(present),
                'p_val'      : round(float(pval), 4),
                'significant': bool(pval < 0.05),
            })
    return pd.DataFrame(detail)


frames = [build_interference_sc(df_all, m) for m in ALL_METHODS]
df_int = pd.concat(frames, ignore_index=True)
df_int.to_csv(OUT_DIR / 'scorecam_interference_comparison.csv', index=False)

print('Significant interference pairs per method (p<0.05):')
for mkey in ALL_METHODS:
    sub = df_int[df_int['method'] == METHOD_LABELS[mkey]]
    sig = sub['significant'].sum()
    print(f'  {METHOD_LABELS[mkey]:<28}: {sig}/{len(sub)}')

## 15. Final Summary and Comparison CSV

In [ ]:
print('=' * 70)
print('SCORE-CAM vs GRADIENT-BASED METHODS — MoNuSAC 2020')
print('=' * 70)

print(f'\n(1) Macro IoU@0.50 (n={len(df_all)} test images):')
for mkey in ALL_METHODS:
    row = df50[df50['method'] == METHOD_LABELS[mkey]]
    if not row.empty:
        tag = '[gradient-free]' if mkey == 'scorecam' else '[gradient-based]'
        print(f'  {METHOD_LABELS[mkey]:<28}: '
              f'{row["macro_iou"].values[0]:.4f}  {tag}')

print(f'\n(2) Gap Score-CAM vs gradient methods:')
sc_m = df50[df50['method'] == 'Score-CAM']['macro_iou'].values[0]
for gkey in GRAD_METHODS:
    gm = df50[df50['method'] == METHOD_LABELS[gkey]]['macro_iou'].values[0]
    print(f'  vs {METHOD_LABELS[gkey]:<26}: {sc_m - gm:+.4f}')

print(f'\n(3) Cardinality monotonicity (Spearman):')
for mkey, (rho, p) in spearman_results.items():
    print(f'  {METHOD_LABELS[mkey]:<28}: rho={rho:+.4f}  p={p:.4f}')

print(f'\n(4) Interference (significant pairs on test set):')
for mkey in ALL_METHODS:
    sub = df_int[df_int['method'] == METHOD_LABELS[mkey]]
    sig = sub['significant'].sum()
    print(f'  {METHOD_LABELS[mkey]:<28}: {sig}/{len(sub)}')

# Save comparison CSV
comp_rows = []
for mkey in ALL_METHODS:
    row50 = df50[df50['method'] == METHOD_LABELS[mkey]]
    rp    = spearman_results.get(mkey, (float('nan'), float('nan')))
    sub_i = df_int[df_int['method'] == METHOD_LABELS[mkey]]
    comp_rows.append({
        'method'            : METHOD_LABELS[mkey],
        'family'            : 'gradient-free' if mkey == 'scorecam' else 'gradient-based',
        'macro_iou_50'      : row50['macro_iou'].values[0] if not row50.empty else float('nan'),
        'spearman_rho'      : round(rp[0], 4),
        'spearman_p'        : round(rp[1], 4),
        'n_sig_interference': int(sub_i['significant'].sum()) if not sub_i.empty else 0,
        'n_pairs_tested'    : len(sub_i),
    })
df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(OUT_DIR / 'scorecam_vs_gradient_comparison.csv', index=False)
print(f'\nSaved → scorecam_vs_gradient_comparison.csv')

print('\nFPN-ScoreCAM: deferred to future work.')
print('  Same reasons as PanNuke: early-layer channel masking requires')
print('  full downstream propagation; cannot post-hoc blend 3 layers.')
print('  The MoNuSAC pyramid reversal finding is already established')
print('  by 4 gradient-based FPN methods.')

## 16. Output Summary

In [ ]:
print('=' * 65)
print('OUTPUT FILES — monusac_scorecam/')
print('=' * 65)
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(OUT_DIR)):<55}  '
              f'{f.stat().st_size/1024:>7.1f} KB')
n_trip = sum(1 for _ in TRIP_DIR.glob('*.png'))
print(f'\nTriplet figures: {n_trip}  ({N_TEST} test images)')